In [1]:
from friepedia import friepedia_webinars

In [3]:
engine = friepedia_webinars(api_key="FRIE-KENOELLE")

In [4]:
engine.run()

Please Switch the Google Colab Theme to Light Mode for the best experience!


Output()

In [ ]:
print("\n[Test 3] Testing Schema/Column awareness...")
print(f"Available NLP Columns: {engine.nlp_columns}")

print("\n✅ Technical Tests Completed.")

In [ ]:
test_scenarios = [
    {"query": "Digital Banking", "column": "Summary", "top_n": 3},
    {"query": "Cybersecurity", "column": "Title", "top_n": 5},
    {"query": "IKN Infrastructure", "column": "Important Figures/Statistics", "top_n": 1},
    {"query": "API Integration", "column": "Summary", "top_n": 6}
]

master_results = []

In [ ]:
print(f"🚀 Running {len(test_scenarios)} test scenarios...")

for test in test_scenarios:
    payload = {"api_key": engine.api_key, "query": test["query"], "top_k": test["top_n"]}
    try:
        res = requests.post(f"{engine.base_url}/v1/webinars/search", json=payload)
        if res.status_code == 200:
            df = pd.DataFrame(res.json())
            df['test_query_source'] = test['query'] # Discovery Audit Trail
            master_results.append(df)
            
            # Inline Validation
            v_check = "✅" if any(c.startswith("vector_") for c in df.columns) else "❌"
            print(f"  - '{test['query']}': Found {len(df)} rows | Vectors: {v_check}")
    except Exception as e:
        print(f"  - '{test['query']}': ❌ Connection Error: {e}")

In [ ]:
# 3. First-Query Discovery Cleanup
if master_results:
    final_export_df = pd.concat(master_results, ignore_index=True)
    # Normalize titles to catch hidden duplicates
    final_export_df['Title'] = final_export_df['Title'].astype(str).str.strip()
    # Keep only the first time a webinar is found
    final_export_df = final_export_df.drop_duplicates(subset=['Title'], keep='first').reset_index(drop=True)
    
    print(f"\n✨ Cleanup Complete: {len(final_export_df)} unique webinars identified.")

In [ ]:
if 'final_export_df' in locals() and not final_export_df.empty:
    # Exporting artifacts
    final_export_df.to_csv("Friepedia_Stress_Test_Results.csv", index=False)
    final_export_df.to_excel("Friepedia_Stress_Test_Results.xlsx", index=False)
    
    print(f"📂 Documentation Exported: 9 Unique Rows saved to CSV and XLSX.")
    print(f"📊 Previewing Top 5 Entries:")
    display(final_export_df[['Title', 'test_query_source']].head(5))
else:
    print("❌ Error: No data available to export.")